# Workshop 3 Lab — Beginner
### Computation 101 · IQ Biology Bootcamp 2026

You're about to reconstruct a real RNA-seq analysis in Google Colab — a full Linux computer in your
browser. **Run each cell in order** (Shift+Enter). Where you see `...`, replace it with your code before running.

*Background:* **ZFP36L2** is an RNA-binding protein that grabs certain messenger RNAs and marks them for
destruction. Knock the gene out in a mouse (**KO**), and those mRNAs are no longer destroyed — so they go
**up**. Researchers measured this in six tissues with RNA-seq, then used DESeq2 to score every gene. A gene
counts as **up-regulated** when its `log2FoldChange > 1` *and* `padj < 0.05`.

**When a cell turns red — read it, don't panic.** Python puts the real problem on the *last* line:
`KeyError: 'padj'` means there's no column by that name; `NameError: name 'up' is not defined` means you
skipped the cell that made `up`. Read bottom-up, fix that one thing, rerun. Still stuck after a couple
tries? **Bring it to the workshop — that's exactly what it's for.**

In [ ]:
# One-time setup. The lines starting with ! are REAL shell commands running in this
# Colab Linux machine — the same commands you'd type on Fiji. This clones the course
# repo (only if it isn't here yet) and points DATA at the ZFP36L2 dataset inside it.
![ -d iqbio-computation-101-2026 ] || git clone --depth 1 https://github.com/gsstephenson/iqbio-computation-101-2026.git
import pandas as pd, os
DATA = 'iqbio-computation-101-2026/data/zfp36l2'
print('data folder contains:', os.listdir(DATA))

## Your job: one tissue
Pick a single tissue — **bone marrow** — and count how many genes go significantly **up** when
ZFP36L2 is knocked out. One tissue is one clue; later levels combine all six.

In [ ]:
# Load bone marrow's DESeq2 table. Each row is a gene; index_col=0 makes the gene ID the row label.
bm = pd.read_csv(os.path.join(DATA, 'de', ...), index_col=0)   # bone marrow's table is 'BM_DE.csv' (pattern: <TISSUE>_DE.csv)
bm.head()

The columns that matter: **`log2FoldChange`** (how much the gene moved; positive = up) and
**`padj`** (significance; smaller = more confident). **`Gene.name`** is the human-readable symbol.

In [ ]:
# Keep only genes that are strongly AND significantly up: log2FoldChange > 1 and padj < cutoff.
up = bm[(bm['log2FoldChange'] > 1) & (bm['padj'] < ...)]   # the standard significance cutoff
print(len(up), 'genes up in bone marrow')

Over a thousand genes — bone marrow is the tissue ZFP36L2 loss hits hardest. Let's see the biggest movers.

In [ ]:
# Sort by fold change, largest first, and show the top 10 with their symbols.
up.sort_values(..., ascending=False)[['log2FoldChange', 'padj', 'Gene.name']].head(10)

**You just filtered a real DESeq2 table.** One tissue done. The real question is bigger: is there a gene
that's up in *every* tissue? That's the **Intermediate** workbook.

## See it — your first plot
Numbers are hard to feel; a picture isn't. You have `up`, bone marrow's up-regulated genes. Draw the ten
biggest movers as a horizontal bar chart. (In Colab, matplotlib is already installed — just import it.)

In [ ]:
import matplotlib.pyplot as plt
top = up.sort_values('log2FoldChange', ascending=False).head(10)
plt.figure(figsize=(6, 4))
plt.barh(top['Gene.name'], top[...])          # which column is the fold change?
plt.xlabel('log2 fold change'); plt.title('Top 10 up in bone marrow')
plt.gca().invert_yaxis(); plt.tight_layout(); plt.show()

## So what? — the question that comes next
You should see ~1343 genes up in bone marrow — if your count differs, re-check your padj cutoff and the > direction. A
bioinformatician's *very next* question is: what do they DO — which biological processes are over-represented?
That's **functional enrichment**, and you'll run it at the Advanced
level. For now, notice a picture already beat a list — which is exactly why you plotted.

## One last lesson: the statistic won't save you
You just drew pictures of your data. Here's *why* that matters — a trick every bioinformatician meets once.
Four tiny datasets. First, the numbers:

In [ ]:
import numpy as np, matplotlib.pyplot as plt
x123 = [10,8,13,9,11,14,6,4,12,7,5]; x4 = [8,8,8,8,8,8,8,19,8,8,8]
quartet = {'I':(x123,[8.04,6.95,7.58,8.81,8.33,9.96,7.24,4.26,10.84,4.82,5.68]),
           'II':(x123,[9.14,8.14,8.74,8.77,9.26,8.10,6.13,3.10,9.13,7.26,4.74]),
           'III':(x123,[7.46,6.77,12.74,7.11,7.81,8.84,6.08,5.39,8.15,6.42,5.73]),
           'IV':(x4,[6.58,5.76,7.71,8.84,8.47,7.04,5.25,12.50,5.56,7.91,6.89])}
for k,(x,y) in quartet.items():
    x=np.array(x,float); y=np.array(y,float); m,b=np.polyfit(x,y,1)
    print(f'{k:3}  mean_x={x.mean():.1f}  mean_y={y.mean():.2f}  corr={np.corrcoef(x,y)[0,1]:.3f}  fit: y={m:.2f}x+{b:.2f}')

Identical — same means, near-identical correlation (≈0.816), the same best-fit line. By the numbers, one dataset.
Now **look**:

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(7, 6))
for ax,(k,(x,y)) in zip(axes.ravel(), quartet.items()):
    x=np.array(x,float); y=np.array(y,float); m,b=np.polyfit(x,y,1)
    ax.scatter(x, y, c='crimson'); ax.plot([2,20],[m*2+b, m*20+b], 'k--', lw=0.8)
    ax.set_title(f'Set {k}'); ax.set_xlim(2,20); ax.set_ylim(2,14)
plt.tight_layout(); plt.show()

Same statistics, four different truths: **I** a real linear trend; **II** an obvious curve (a straight line
is the *wrong model*); **III** a clean line wrecked by one **outlier**; **IV** a single high-leverage point
that invents the whole correlation. This is **Anscombe's Quartet** (1973) — and it is the entire reason you
plot. The number never warned you; you had to look.

It's also why RNA-seq lives in **log2 space** and why DESeq2 normalizes by median-of-ratios instead of raw
ratios: choose the right transform, then *validate by eye* with a volcano or MA plot — exactly what you just
did. Don't hedge the statistic. Look at your data, pick the right normalization, and trust the picture.

## One more thing

The dataset you just analyzed isn't a textbook toy. It's a real, published study —
*The RNA binding protein ZFP36L2 displays tissue-selective mRNA targeting in mice*, **RNA Biology (2026)** —
and you re-derived its headline result from the same deposited data its authors released.

Who wrote it? You'll find out the way a scientist does — by looking it up and **citing it properly**. That's
the final act of the bootcamp: in **Workshop 5** you'll pull this paper, build a real citation, and add it to
your repo. Read the author list closely when you get there. Then — next year, someone reproduces *yours*.